# Train models from models/

Run from the repository root or `train` folder. Choose `MODEL_NAME` to train one model, or `None` for all three sequentially. Settings come from `train/cfg/v3.yaml`; checkpoints and metrics are saved per run.

On RunPod, use CUDA-enabled PyTorch and upload the dataset images and labels. See [README.md](README.md) for setup and [SOURCES.md](custom_modules/SOURCES.md) for the experimental blocks and aliases. Knowledge distillation is not included.


## Install the custom environment

Locate the repository and keep Ultralytics settings inside the project.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "train":
    ROOT = ROOT.parent
os.environ.setdefault("YOLO_CONFIG_DIR", str(ROOT / ".cache/ultralytics"))

Install the pinned Ultralytics source and dependencies. After the first installation, restart the kernel before Setup.


In [ ]:
subprocess.run([sys.executable, str(ROOT / "train/setup_ultralytics.py")], check=True)
ULTRALYTICS_SOURCE = ROOT / "vendor/ultralytics"
%pip install -e "{ULTRALYTICS_SOURCE}" "timm==1.0.31" wandb
print("After the first installation, restart the kernel before running Setup.")

## Setup

Run these cells in order. To train another model, change `MODEL_NAME` and rerun the remaining Setup cells, architecture check, and training. Each setup preserves earlier runs in separate directories.


Locate the repository after a kernel restart.


In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "train":
    ROOT = ROOT.parent
os.environ.setdefault("YOLO_CONFIG_DIR", str(ROOT / ".cache/ultralytics"))

Import the training and reporting libraries.


In [ ]:
import csv
import gc
from datetime import datetime
from time import perf_counter

import torch
import ultralytics
import wandb
import yaml
from IPython.display import display
from ultralytics import YOLO, settings

Verify the custom installation. CUDA should report `True` on a GPU RunPod.


In [ ]:
if not Path(ultralytics.__file__).resolve().is_relative_to(ROOT / "vendor/ultralytics"):
    raise RuntimeError("Run the installation cell and restart the kernel to use the custom Ultralytics checkout.")
print("CUDA available:", torch.cuda.is_available())

Choose one model, or set `MODEL_NAME = None` for all three. Keep the space in the ADown filename.


In [ ]:
MODEL_NAME = "yolov8_baseline.yaml"
# MODEL_NAME = "ADown _C2fFaster.yaml"
# MODEL_NAME = "GSConv_VoV-GSCP1.yaml"
# MODEL_NAME = None

WANDB_PROJECT = "cone-detection"
CONFIG = ROOT / "train/cfg/v3.yaml"

Check that the selected architecture file exists.


In [ ]:
MODEL_PATHS = sorted(
    path for path in (ROOT / "models").iterdir()
    if path.is_file() and path.suffix.lower() in {".yaml", ".yml"}
)
if not MODEL_PATHS:
    raise FileNotFoundError(f"No model architecture YAMLs in {ROOT / 'models'}")
if MODEL_NAME is not None:
    available_names = [path.name for path in MODEL_PATHS]
    if MODEL_NAME not in available_names:
        raise ValueError(f"Unknown MODEL_NAME {MODEL_NAME!r}. Choose one of {available_names}, or None for all.")
    MODEL_PATHS = [ROOT / "models" / MODEL_NAME]

Load training and export settings from `v3.yaml`. Keep settings consistent across comparisons; use a fixed batch size if needed. See [README.md](README.md) for worker settings.


In [ ]:
train_args = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
export_args = {key: train_args.pop(key) for key in ("format", "half", "int8", "nms")}
IMGSZ = train_args["imgsz"]

Create a timestamped output directory for this run.


In [ ]:
experiment_name = train_args.pop("name", None) or "comparison"
RUN_PROJECT = ROOT / train_args["project"] / f"{experiment_name}_{datetime.now():%Y%m%d_%H%M%S_%f}"
RUN_PROJECT.mkdir(parents=True, exist_ok=False)
train_args["project"] = str(RUN_PROJECT)

Read the dataset YAML and resolve its local root.


In [ ]:
SOURCE_DATA = ROOT / train_args["data"]
dataset_config = yaml.safe_load(SOURCE_DATA.read_text(encoding="utf-8"))
DATASET_ROOT = SOURCE_DATA.parent.resolve()
dataset_config["path"] = str(DATASET_ROOT)

Verify the training and validation paths. Upload missing images and labels before continuing.


In [ ]:
for split in ("train", "val"):
    paths = dataset_config[split]
    for split_path in paths if isinstance(paths, list) else [paths]:
        if not (DATASET_ROOT / split_path).exists():
            raise FileNotFoundError(f"Missing {split} data: {DATASET_ROOT / split_path}. Upload the dataset first.")

Save a portable dataset YAML for this run, preserving the original.


In [ ]:
DATA = RUN_PROJECT / "data.yaml"
DATA.write_text(yaml.safe_dump(dataset_config, sort_keys=False), encoding="utf-8")
train_args["data"] = str(DATA)


Define a helper to save `comparison.csv` after each model.


In [ ]:
SUMMARY_PATH = RUN_PROJECT / "comparison.csv"
SUMMARY_FIELDS = ["model", "best_weights", "training_seconds", "validation_seconds", "mAP50", "mAP50-95"]


def save_comparison(records):
    with SUMMARY_PATH.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=SUMMARY_FIELDS)
        writer.writeheader()
        writer.writerows(records)

Review the selected models and output directory.


In [ ]:
print("Models:", *(path.name for path in MODEL_PATHS), sep="\n  ")
print("Outputs:", RUN_PROJECT)

## Check selected architectures

Build each selected model before training. Resolve any reported errors first. For a short training and checkpoint test, see [README.md](README.md).


In [ ]:
# Clear approval on every attempt, including attempts that fail partway through.
CHECKED_MODEL_PATHS = None
load_errors = []
for model_path in MODEL_PATHS:
    try:
        candidate = YOLO(str(model_path), task="detect")
        print(f"OK: {model_path.name}")
        del candidate
    except Exception as error:
        load_errors.append(f"{model_path.name}: {type(error).__name__}: {error}")
    finally:
        gc.collect()

if load_errors:
    raise RuntimeError(
        "Training has not started. These architectures could not load:\n  "
        + "\n  ".join(load_errors)
        + "\nRerun the installation cell, restart the kernel, and rerun Setup and this check."
    )
CHECKED_MODEL_PATHS = tuple((str(path), path.read_bytes()) for path in MODEL_PATHS)
print("All architectures loaded successfully. Ready to train.")


## Train selected models

Train from scratch with validation enabled. Run these cells in order; the training loop starts the run.


Sign in to W&B for a separate logging run per model.


In [ ]:
settings.update({"wandb": True})
wandb.login()

Define the training function. It collects metrics and releases model memory; training has not started yet.


In [ ]:
def train_one_model(model_path):
    run_name = model_path.stem.replace(" ", "_")
    start = perf_counter()
    print(f"Training {model_path.name}")
    with wandb.init(project=WANDB_PROJECT, name=run_name, dir=str(RUN_PROJECT)):
        model = YOLO(str(model_path), task="detect")
        results = model.train(**{**train_args, "name": run_name})
        record = {
            "model": model_path.stem,
            "best_weights": str(model.trainer.best),
            "training_seconds": round(perf_counter() - start, 1),
        }
        if results is not None:
            record.update({"mAP50": float(results.box.map50), "mAP50-95": float(results.box.map)})
    del model, results
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return record

Start training and save metrics after each model. This starts fresh training; rerun Setup first for a new output directory.


In [ ]:
if globals().get("CHECKED_MODEL_PATHS") != tuple((str(path), path.read_bytes()) for path in MODEL_PATHS):
    raise RuntimeError("Run the architecture check successfully after selecting or editing models.")

run_records = []
best_weights = {}
suite_start = perf_counter()
for model_path in MODEL_PATHS:
    record = train_one_model(model_path)
    best_weights[model_path.stem] = Path(record["best_weights"])
    run_records.append(record)
    save_comparison(run_records)
    print(record)

Show training time and comparison records.


In [ ]:
print(f"Training total: {(perf_counter() - suite_start) / 3600:.2f} hours")
print("Comparison:", SUMMARY_PATH)
display(run_records)

## Select a checkpoint

Default to the first model trained in this run. Change `SELECTED_MODEL` for another model, or set `BEST_WEIGHTS` to an earlier checkpoint after Setup. Custom checkpoints require the same layer definitions.


In [ ]:
SELECTED_MODEL = next(iter(best_weights))
BEST_WEIGHTS = best_weights[SELECTED_MODEL]
best_model = YOLO(str(BEST_WEIGHTS), task="detect")
print(BEST_WEIGHTS)


## Separate validation (optional)

Reevaluate each best checkpoint and update `comparison.csv`. Use `split="test"` for final evaluation and save those results separately.


In [ ]:
for record in run_records:
    start = perf_counter()
    validation_model = YOLO(record["best_weights"], task="detect")
    metrics = validation_model.val(
        data=str(DATA), split="val", imgsz=IMGSZ,
        workers=train_args["workers"],
        project=str(Path(record["best_weights"]).parent.parent), name="validation",
    )
    record.update({
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
        "validation_seconds": round(perf_counter() - start, 1),
    })
    save_comparison(run_records)
    print(record)
    del validation_model, metrics
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Review the validation metrics saved to the comparison CSV.


In [ ]:
display(run_records)

## TensorRT export

Export FP16 `best.engine` beside `best.pt` using `v3.yaml` and batch size 1. Requires an NVIDIA GPU and [compatible TensorRT](https://docs.ultralytics.com/integrations/tensorrt/); build on deployment-compatible hardware.

For calibrated INT8, set `half: False` and `int8: True`.


In [ ]:
start = perf_counter()
ENGINE_PATH = Path(best_model.export(
    **export_args,
    data=str(DATA),
    imgsz=IMGSZ,
    batch=1,
    device=0,
))
print(f"Engine: {ENGINE_PATH}")
print(f"Export: {perf_counter() - start:.1f} s")


## Image prediction

Choose `IMAGE_PATH`; the default uses a test image.


In [ ]:
IMAGE_PATH = next((DATASET_ROOT / "test/images").glob("*.jpg"))

Predict with the selected engine and save the annotated image.


In [ ]:
start = perf_counter()
engine = YOLO(str(ENGINE_PATH), task="detect")
predictions = engine.predict(
    source=str(IMAGE_PATH),
    imgsz=IMGSZ,
    conf=0.4,
    device=0,
    save=True,
    project=str(BEST_WEIGHTS.parent.parent),
    name="predictions",
)

Display the prediction and timing, including engine loading.


In [ ]:
print(f"Prediction (including engine loading): {perf_counter() - start:.1f} s")
for prediction in predictions:
    display(prediction.plot(pil=True))

## TensorRT benchmark

Evaluate the engine on validation images with batch size 1 and square inputs.


In [ ]:
start = perf_counter()
engine = YOLO(str(ENGINE_PATH), task="detect")
benchmark_metrics = engine.val(
    data=str(DATA),
    split="val",
    imgsz=IMGSZ,
    batch=1,
    conf=0.001,
    device=0,
    workers=train_args["workers"],
    rect=False,
    plots=False,
)

Show accuracy, timing, and inference FPS. FPS excludes preprocessing and postprocessing.


In [ ]:
print(f"mAP50-95: {benchmark_metrics.box.map:.4f}")
print("Milliseconds per image:", benchmark_metrics.speed)
print(f"Inference FPS: {1000 / benchmark_metrics.speed['inference']:.1f}")
print(f"Benchmark: {perf_counter() - start:.1f} s")